# 🛡️ Type Safe to Her — User Guide

This tool uses **TypeSafe AI** to evaluate text messages sent in relationships. It scores proposed text messages across structured output decision types (`noul`, `score`, and `choice`) to prevent unintended arguments or misunderstandings.

---

## 📊 Evaluation Criteria Breakdown

### 1. `noul` Criteria (Continuous Probability: 0.0 to 1.0)
* **`dismissiveness_risk`**: Measures if the reply is low-effort or uses one-word brush-offs (`k`, `fine`, `whatever`).
* **`trap_answer_risk`**: Evaluates whether a question requires emotional reassurance, but the draft responds with literal logic.
* **`sarcasm_leak_risk`**: Detects if playful jokes or dry humor will sound mean or condescending when read without tone of voice.

### 2. `score` Criteria (Numeric Scale: 1 to 10)
* **`couch_sleep_index`**: Rates overall risk from **1** (Comfy bed together) to **10** (Sleeping on the couch tonight).
* **`emotional_effort_score`**: Rates effort and affection from **1** (Robotic/cold) to **10** (Golden Retriever warmth).

### 3. `choice` Criteria (Categorical Verdicts)
* **`perceived_emotion`**: How she will likely interpret your mood (`Loving`, `Passive-Aggressive`, `Ignoring me`, `Overly Logical`).
* **`overall_judgement`**: `🟢 GREEN LIGHT`, `🟡 YELLOW LIGHT`, or `🔴 RED LIGHT`.
* **`actionable_advice`**: Recommended correction strategy before sending.

---

## ⚔️ Comparison Examples: Safe vs. Unsafe Answers

### Scenario 1: The Loaded Question
> **Her Question:** *"Be honest, do you think I've gained weight recently?"*

* 🔴 **UNSAFE (High Danger / Trap Risk):**
  > `"A little bit, but honestly you still look fine to me and we can go to the gym together!"`
  * **Evaluation:** High `trap_answer_risk` (0.98), High `couch_sleep_index` (10/10), `RED LIGHT`. (Fails to realize this is an insecurity check requiring reassurance, not an analytical health evaluation).

* 🟢 **SAFE:**
  > `"What?? You look absolutely amazing as always! Stop overthinking, I love you so much ❤️"`
  * **Evaluation:** Low `trap_answer_risk` (0.02), High `emotional_effort_score` (9/10), `GREEN LIGHT`.

---

### Scenario 2: What to Eat for Dinner
> **Her Question:** *"What do you want to eat tonight?"*

* 🔴 **UNSAFE (Dismissive Risk):**
  > `"Whatever, I don't care, you pick."`
  * **Evaluation:** High `dismissiveness_risk` (0.88), Perceived as: *He's uninterested*, `YELLOW/RED LIGHT`.

* 🟢 **SAFE:**
  > `"I'm down for anything! How about sushi or Mexican? I can pick it up on my way back ❤️"`
  * **Evaluation:** Low `dismissiveness_risk` (0.05), High `emotional_effort_score` (8/10), `GREEN LIGHT`.

---

### Scenario 3: Bad Day at Work
> **Her Statement:** *"My boss gave me extra work 5 minutes before logging off, I'm so annoyed!"*

* 🔴 **UNSAFE (Overly Logical):**
  > `"Did you try telling him that your shift was over? You need to set better boundaries at work."`
  * **Evaluation:** "Of course I know what to do at work, I just want you to comfort me, but you can't even mind to say some sweet word", Low empathy, `YELLOW LIGHT`.

* 🟢 **SAFE:**
  > `"That is so unfair! You worked so hard today. Come home, I'll make tea and you can talk to me about it."`
  * **Evaluation:** High `emotional_effort_score` (10/10), Perceived as: *Loving & Attentive*, `GREEN LIGHT`.

In [5]:
!pip install typesafe-sdk

In [21]:
import os
import getpass
from typesafe_sdk import TypeSafeClient

# 1. Initialize Client
if "TYPESAFE_API_KEY" not in os.environ:
    os.environ["TYPESAFE_API_KEY"] = getpass.getpass("Enter your TypeSafe API Key: ")

client = TypeSafeClient()

def analyze_text_safety(draft_message: str, relationship_context: str = "Standard conversation"):
    """
    Advanced TypeSafe engine to evaluate relationship text message hazards.
    """
    state = f"""
    Draft message to girlfriend: "{draft_message}"
    Relationship context: {relationship_context}
    """

    questions = {
        # NOUL (Continuous Probability 0.0 - 1.0)
        "dismissiveness_risk": {
            "type": "noul",
            "instructions": "Probability this text is low-effort, dismissive, or uses dangerous single-word replies (e.g. 'k', 'fine', 'sure')."
        },
        "trap_answer_risk": {
            "type": "noul",
            "instructions": "Probability this message attempts to give a logical answer to a loaded/trap question where emotional validation was required instead."
        },

        # SCORE (Ordered rubric scale with criteria array from lowest to highest)
        "couch_sleep_index": {
            "type": "score",
            "instructions": "Estimate how much trouble this text will cause in the relationship.",
            "criteria": [
                "1 - Completely safe and cozy together",
                "3 - Slight annoyance or eye roll",
                "5 - Minor argument, needs an immediate follow-up text",
                "8 - Very upset, highly awkward night ahead",
                "10 - Maximum danger, sleeping on the couch tonight"
            ]
        },
        "emotional_effort_score": {
            "type": "score",
            "instructions": "Rate the level of warmth, care, and affection shown in this message.",
            "criteria": [
                "1 - Cold, robotic, zero effort",
                "3 - Neutral or minimal effort",
                "5 - Average, standard response",
                "8 - Sweet, thoughtful, and attentive",
                "10 - Golden Retriever energy, exceptionally loving"
            ]
        },

        # CHOICE (Categorical Verdicts with criteria mapping each key to its definition)
        "perceived_emotion": {
            "type": "choice",
            "instructions": "How will she likely interpret your mood from this message?",
            "criteria": {
                "loving": "She feels loved, valued, and appreciated.",
                "passive_aggressive": "She thinks you are mad at her or being passive-aggressive.",
                "uninterested": "She thinks you are ignoring her or don't care.",
                "overly_logical": "She feels you are being overly analytical instead of comforting her.",
                "playful": "She understands the humor and finds it sweet or funny."
            }
        },
        "overall_judgement": {
            "type": "choice",
            "instructions": "What is the overall safety recommendation for sending this text?",
            "criteria": {
                "green_light": "Safe to send immediately.",
                "yellow_light": "Kind of Risky - add emojis, warmth, or an extra sentence before sending.",
                "red_light": "DANGER - Put the phone down and rewrite immediately."
            }
        },
        "actionable_advice": {
            "type": "choice",
            "instructions": "What is the best immediate corrective action?",
            "criteria": {
                "send_as_is": "Send as is.",
                "add_affection": "Add a sweet nickname or heart emoji to clarify tone.",
                "offer_food": "Agree with her and offer to bring her food or a treat.",
                "delete text": "Apologize directly without over-explaining."
            }
        }
    }

    # Run TypeSafe decision model
    res = client.system_one(state=state, questions=questions).answers

    # Format output using object attribute access (.choice, .score, .noul)
    print(f"\n{'='*55}")
    print(f"📱 DRAFT: \"{draft_message}\"")
    print(f"📋 CONTEXT: {relationship_context}")
    print(f"{'-'*55}")
    print(f"Judgement:            {res['overall_judgement'].choice.upper()}")
    print(f"Couch Sleep Index:    {res['couch_sleep_index'].score}")
    print(f"Emotional Effort:     {res['emotional_effort_score'].score}")
    print(f"Perceived Emotion:    {res['perceived_emotion'].choice}")
    print(f"Trap Answer Risk:     {res['trap_answer_risk'].noul:.1%}")
    print(f"Dismissive Risk:      {res['dismissiveness_risk'].noul:.1%}")
    print(f"Recommended Action:   {res['actionable_advice'].choice}")
    print(f"{'='*55}\n")

# =====================================================================
# TEST SUITE: UNSAFE VS. SAFE RESPONSES
# =====================================================================

def run_scenario_comparison(scenario_title, context, bad_draft, good_draft):

    print("\n❌ --- UNSAFE RESPONSE ---")
    analyze_text_safety(draft_message=bad_draft, relationship_context=context)

    print("\n✅ --- SAFE RESPONSE ---")
    analyze_text_safety(draft_message=good_draft, relationship_context=context)


# --- SCENARIO 1: The Loaded Question ---
run_scenario_comparison(
    scenario_title="The Loaded Question / Insecurity Trap",
    context="She sent a photo asking: 'Be honest, do you think I've gained weight recently?'",
    bad_draft="A little bit",
    good_draft="What?? You look absolutely stunning as always! Stop overthinking, I love you so much ❤️"
)

# --- SCENARIO 2: Long Venting Text ---
run_scenario_comparison(
    scenario_title="Long Venting Text After a Stressful Day",
    context="She sent 3 paragraphs detailing how overwhelming and toxic her workday was.",
    bad_draft="k",
    good_draft="I'm so sorry you had to deal with all that today! You worked so hard. Come home, take it easy, and I'll make us dinner ❤️"
)

# --- SCENARIO 3: Choosing Dinner ---
run_scenario_comparison(
    scenario_title="Decision Avoidance (What to Eat)",
    context="She asked: 'What do you want to get for dinner tonight?'",
    bad_draft="Whatever, I don't care, you pick.",
    good_draft="I'm down for anything! How about sushi or Mexican? I can pick it up on my way home ❤️"
)

# --- SCENARIO 4: Bad Day at Work ---
run_scenario_comparison(
    scenario_title="Problem Solving vs. Emotional Validation",
    context="She texted: 'My boss gave me a huge task right before logging off, I'm so annoyed!'",
    bad_draft="Did you try setting clear boundaries with him? You shouldn't let people walk over you at work.",
    good_draft="That is so unfair! You've been working so hard all week. Treat yourself to a break, we can talk about it when you're home!"
)

# --- SCENARIO 5: Sharing an Achievement ---
run_scenario_comparison(
    scenario_title="Dry Sarcasm Misinterpretation",
    context="She excitedly texted: 'I finally finished reorganizing the entire kitchen pantry!'",
    bad_draft="Wow, revolutionary idea.",
    good_draft="Babe, that's awesome! You've been wanting to get that done for weeks, super proud of you! 🎉"
)

# --- SCENARIO 6: Hurt Feelings / Apology ---
run_scenario_comparison(
    scenario_title="Apology Attempt",
    context="She expressed that her feelings were hurt because you forgot to call her last night.",
    bad_draft="Sorry you feel that way, but I was really tired yesterday.",
    good_draft="I am so sorry babe! I completely lost track of time and that was totally my bad. You mean the world to me, calling you right now ❤️"
)


❌ --- UNSAFE RESPONSE ---

📱 DRAFT: "A little bit"
📋 CONTEXT: She sent a photo asking: 'Be honest, do you think I've gained weight recently?'
-------------------------------------------------------
Judgement:            YELLOW_LIGHT
Couch Sleep Index:    2.38
Emotional Effort:     0.69
Perceived Emotion:    passive_aggressive
Trap Answer Risk:     72.0%
Dismissive Risk:      65.0%
Recommended Action:   delete text


✅ --- SAFE RESPONSE ---

📱 DRAFT: "What?? You look absolutely stunning as always! Stop overthinking, I love you so much ❤️"
📋 CONTEXT: She sent a photo asking: 'Be honest, do you think I've gained weight recently?'
-------------------------------------------------------
Judgement:            YELLOW_LIGHT
Couch Sleep Index:    1.38
Emotional Effort:     3.32
Perceived Emotion:    loving
Trap Answer Risk:     30.0%
Dismissive Risk:      10.0%
Recommended Action:   send_as_is


❌ --- UNSAFE RESPONSE ---

📱 DRAFT: "k"
📋 CONTEXT: She sent 3 paragraphs detailing how overwhelming